<div align="center">
    <h1>Multi-Layer Perceptrons: Part 2</h1>
    <h3>Hyperparameter Tuning on Real Image Data</h3>
    <hr>
</div>

## Learning Objectives

In Part 1 the MLP solved the spirals. Now it faces real images: CIFAR-10, the same dataset on which k-NN and the linear classifier reached about 34% and 40%. Your implementation is done; this notebook is about **training it well**. By the end, you will:

- Know what each hyperparameter controls and how a bad value shows up in the training curves
- Tune the hyperparameters yourself, keeping a log of every run
- Recognize overfitting and underfitting from training and validation accuracy
- See that the first layer of the MLP learns many image templates, instead of one per class
- Choose the final model on the validation set and evaluate it on the test set only once

## From Toy Data to Real Images

| | **Spirals (Part 1)** | **CIFAR-10 (Part 2)** |
|---|---|---|
| Input features $D$ | 2 | 3072 (32×32×3) |
| Classes $C$ | 3 | 10 |
| Training samples | 2,250 | 45,000 |
| Parameters of the MLP | 511 (with $H_1 = 100$, $H_2 = 2$) | about 636,000 (with $H_1 = 200$, $H_2 = 100$) |
| Best validation accuracy | 100% | around 50% |
| Main difficulty | Non-linear boundaries | Hyperparameters and overfitting |

A run with 200 and 100 hidden neurons and 10 epochs takes only a few seconds on a laptop CPU, so you can afford many experiments.

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['image.interpolation'] = 'nearest'

%load_ext autoreload
%autoreload 2

## Data Preparation

The same data as in the linear classifier notebooks:

- 10 classes of 32×32 color images, split into **45,000 training**, **5,000 validation** and **10,000 test** images
- Every color channel normalized with the mean and standard deviation of the training images
- Every image flattened to a vector of $D = 3072$ numbers: the MLP, like the linear classifier, does not know which pixels are neighbors

In [ ]:
import os
import torch
from utils import load_cifar10, reshape_to_vectors, dataset_stats, normalize_per_channel

notebook_dir = os.getcwd()
dataset_path = os.path.join(notebook_dir, 'data', 'datasets', 'CIFAR10')

# download the CIFAR-10 dataset
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10(directory=dataset_path,
                                                              visualize_samples=True)

# Print out the dataset statistics and visualize a few samples
num_features, num_classes, num_samples = dataset_stats(X_train, y_train, X_val, y_val, X_test, y_test, verbose=True)

# Normalize each color channel with the mean and standard deviation of the training images
X_train, X_val, X_test = normalize_per_channel(X_train, X_val, X_test)

X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).long()

X_val = torch.from_numpy(X_val).float()
y_val = torch.from_numpy(y_val).long()

X_test = torch.from_numpy(X_test).float()
y_test = torch.from_numpy(y_test).long()

# Reshape the images to vectors
X_train, X_val, X_test = reshape_to_vectors(X_train, X_val, X_test)

## The Hyperparameters

Parameters (the weights and biases) are learned by gradient descent. **Hyperparameters** are the choices you make before training. The guidance below was measured on this dataset with the other hyperparameters at reasonable values.

| Hyperparameter | What it controls | Try | Too small | Too large |
|---|---|---|---|---|
| `hidden_dim_1`, `hidden_dim_2` | Size of the hidden layers (capacity) | 50–500 and 20–200 | Underfits: training accuracy stays low | Slower, and overfits more easily |
| `learning_rate` | Step size of gradient descent | 1e-3 to 1 | Loss decreases very slowly | Loss explodes and becomes `nan` |
| `reg` | Strength of the L2 penalty on the weights | 0 to 1e-1 | Overfits: training accuracy far above validation | Underfits: the network cannot learn |
| `batch_size` | Training images per step | 32–512 | Noisy steps | Every step costs more time |
| `num_epochs` | Number of passes over the training set | 5–100 | Stops before the validation accuracy levels off | Costs time; overfits if `reg` is small |
| `activation` | The non-linearity from Part 1 | `'relu'`, `'tanh'`, `'sigmoid'` | | |

Some things to keep in mind:

- **Epochs.** One **epoch** is one pass over all 45,000 training images in a new random order, e.g. 225 gradient-descent steps with batches of 200. After every epoch, `train` measures the loss and the accuracies and shows them in the progress bar.
- **Best checkpoint.** `train` keeps the parameters with the best validation accuracy, so training longer than needed costs time but does not make the returned model worse.
- **The plotted loss includes the regularization.** It is the cross-entropy plus `reg` times the sum of squared weights. With about 636,000 parameters the penalty is large at the start: with `reg = 1e-3` the loss starts around 3.7, and with `reg = 1e-1` above 60. Even the cross-entropy alone starts around 3 rather than $\ln 10 \approx 2.3$, because the random initial weights already favor some classes.
- **Reading the curves.** Training and validation accuracy rising together: keep going. Training accuracy far above validation: overfitting, so increase `reg` or stop earlier. Both low: underfitting, so train longer, use a larger network or a better learning rate.

## An Experiment Log

Tuning means many runs, and it is easy to lose track of them. The two helpers below keep a log:

- `run_experiment(name, ...)` trains one MLP with the given hyperparameters and stores the model, its training curves and its accuracies in the dictionary `experiments` under `name`. Remember that `train` returns the parameters from the **best epoch** (the one with the highest validation accuracy), not from the last one; the log shows which epoch that was.
- `show_experiments(compare=[...])` prints all runs as a table sorted by validation accuracy, and plots the training curves of the runs you list in `compare`.

Every run starts from the same random seed, so two runs differ only in their hyperparameters. A progress bar shows the epochs together with the current validation loss and the training and validation accuracy.

**GPU.** If your computer has an NVIDIA GPU (CUDA) or an Apple silicon chip (MPS), the training runs there; otherwise on the CPU. The trained model is moved back to the CPU afterwards. For the network sizes in this notebook the CPU is about as fast; a GPU pays off for large layers (thousands of neurons).

In [ ]:
import time
from sklearn.metrics import accuracy_score

from assignments import MLPClassifier
from utils import plot_training, plot_training_runs

# Train on a GPU if there is one: CUDA (NVIDIA) or MPS (Apple silicon), otherwise on the CPU
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Training on: {device}")

# Every run is stored here: name -> hyperparameters, trained model, curves and accuracies
experiments = {}


def run_experiment(name: str, hidden_dim_1: int, hidden_dim_2: int, learning_rate: float,
                   reg: float, batch_size: int, num_epochs: int, activation: str = 'relu') -> MLPClassifier:
    """Train one MLP, store it in `experiments` and print its accuracies."""
    torch.random.manual_seed(69)
    model = MLPClassifier(num_features, hidden_dim_1, hidden_dim_2, num_classes,
                          activation=activation, reg=reg, learning_rate=learning_rate,
                          batch_size=batch_size, num_epochs=num_epochs, device=device)

    start = time.perf_counter()
    loss_history, acc_history = model.train(X_train, y_train, X_val, y_val)
    train_time = time.perf_counter() - start

    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))

    # train() returns the parameters from the epoch with the best validation accuracy
    best_epoch = max(acc_history['val'], key=acc_history['val'].get)

    experiments[name] = dict(
        hyperparameters=dict(hidden_dim_1=hidden_dim_1, hidden_dim_2=hidden_dim_2,
                             learning_rate=learning_rate, reg=reg, batch_size=batch_size,
                             num_epochs=num_epochs, activation=activation),
        model=model, loss_history=loss_history, acc_history=acc_history,
        train_acc=train_acc, val_acc=val_acc, best_epoch=best_epoch, time=train_time,
    )
    print(f"{name}: best epoch {best_epoch} of {num_epochs}, training accuracy {train_acc:.1%}, "
          f"validation accuracy {val_acc:.1%} ({train_time:.0f} s)")
    return model


def show_experiments(compare: list = None) -> None:
    """Print all runs sorted by validation accuracy and plot the curves of the runs in `compare`."""
    header = (f"{'name':30}{'H1':>5}{'H2':>5}{'lr':>7}{'reg':>7}{'batch':>6}{'epochs':>7}{'best':>6}"
              f"{'activation':>11}{'train':>8}{'val':>8}{'time':>6}")
    print(header)
    print("-" * len(header))
    for name, run in sorted(experiments.items(), key=lambda item: item[1]['val_acc'], reverse=True):
        h = run['hyperparameters']
        print(f"{name[:29]:30}{h['hidden_dim_1']:>5}{h['hidden_dim_2']:>5}{h['learning_rate']:>7g}"
              f"{h['reg']:>7g}{h['batch_size']:>6}{h['num_epochs']:>7}{run['best_epoch']:>6}{h['activation']:>11}"
              f"{run['train_acc']:>8.1%}{run['val_acc']:>8.1%}{run['time']:>5.0f}s")

    if compare:
        plot_training_runs({name: experiments[name]['loss_history'] for name in compare},
                           {name: experiments[name]['acc_history'] for name in compare}, xlabel="Epochs")

## Worked Example: The Learning Rate

The learning rate is usually the first hyperparameter to tune. We train the same network (200 and 100 hidden neurons, `reg = 1e-3`, batches of 200, 10 epochs) with four learning rates, 10× apart, and compare the curves.

In [ ]:
learning_rates = [1e-3, 1e-2, 1e-1, 1]
for lr in learning_rates:
    run_experiment(f"lr = {lr:g}", hidden_dim_1=200, hidden_dim_2=100, learning_rate=lr,
                   reg=1e-3, batch_size=200, num_epochs=10)

print()
show_experiments(compare=[f"lr = {lr:g}" for lr in learning_rates])

Notice:

- **lr = 0.001** is too small: the validation accuracy is still rising slowly after 10 epochs and ends at 39%.
- **lr = 0.01** learns faster and reaches 48%.
- **lr = 0.1** is the best of the four: the fastest start and 50.5% after 10 epochs.
- **lr = 1** is too large: every step overshoots, the loss grows about 100× per step, reaches $10^{24}$ after 7 steps and then becomes `nan` (its curve ends at the top of the plot). A network with `nan` weights predicts a single class, so the accuracy stays at 10%, no better than the untrained network at epoch 0.

**A loss of `nan` or `inf` almost always means a learning rate that is too large.**

A good learning rate is usually found this way: try values 10× apart, then refine around the best one.

### What Overfitting Looks Like

Let's train the best of these networks for 100 epochs instead of 10 and watch training and validation accuracy.

In [ ]:
run_experiment("lr = 0.1, 100 epochs", hidden_dim_1=200, hidden_dim_2=100, learning_rate=0.1,
               reg=1e-3, batch_size=200, num_epochs=100)

long_run = experiments["lr = 0.1, 100 epochs"]
plot_training(long_run['loss_history'], long_run['acc_history'], ema=True, alpha=0.5, xlabel="Epochs")

Notice:

- The training accuracy keeps climbing to almost 90%, but the validation accuracy peaks at about 52% after 17 epochs and then slowly drops to 50%.
- The **validation loss** is lowest after about 15 epochs and then rises again, while the training loss keeps falling. The network starts to **memorize** the training images instead of learning what the classes have in common.
- `train` returns the parameters from the **best epoch**, not the last one: the returned network has 73.7% training and 52.7% validation accuracy, while the network after epoch 100 had 87.5% and 50.4%. So the returned model is fine, but the last 83 epochs only cost time. A larger `reg` would reduce the gap.

## Your Turn: Tune the MLP

Find hyperparameters with the best **validation** accuracy. Above 50% is a good result for this MLP.

1. Set the hyperparameters in the cell below, give the run a new `experiment_name` and run both cells. Every run is kept in the log.
2. Do **at least 5 runs**. Change one or two hyperparameters at a time, so you can tell what caused a change.
3. Use the table and the curves (`show_experiments`) to decide what to try next.

Some strategies that work:

- **Coarse to fine:** try values 10× apart first (like the learning rates above), then refine around the best one.
- **One at a time:** keep good values fixed and tune the next hyperparameter.
- **Read the curves:** overfitting → more `reg`; underfitting → larger network, longer training or a better learning rate.

Never use the test set while tuning. It is used once, at the end of the notebook.

In [ ]:
# Hyperparameters for the MLP
# TODO: Change the hyperparameters and train the neural network
hidden_dim_1 = None  # the number of hidden units in the first hidden layer (int)
hidden_dim_2 = None  # the number of hidden units in the second hidden layer (int)
reg = None  # the regularization strength (float)
learning_rate = None  # the learning rate (float)
batch_size = None  # the batch size (int)
num_epochs = None  # the number of passes over the training set (int)
activation = 'relu'  # the activation function: 'relu', 'tanh' or 'sigmoid'

In [ ]:
# Give every run a new name, so that the log keeps all of them
experiment_name = "my run 1"

hyperparameters = dict(hidden_dim_1=hidden_dim_1, hidden_dim_2=hidden_dim_2, learning_rate=learning_rate,
                       reg=reg, batch_size=batch_size, num_epochs=num_epochs, activation=activation)

if any(value is None for value in hyperparameters.values()):
    print("Set all hyperparameters in the cell above first.")
else:
    run_experiment(experiment_name, **hyperparameters)

    # Plot the training loss and accuracy of this run
    run = experiments[experiment_name]
    plot_training(run['loss_history'], run['acc_history'], ema=True, alpha=0.5, xlabel="Epochs")

### Compare Your Runs

The table lists every run so far, best validation accuracy first. Put the names of the runs you want to compare into `compare`, for example your best and your worst run.

In [ ]:
show_experiments(compare=["lr = 0.1", "lr = 1"])

## Diagnosing the Best Model

We take the run with the best validation accuracy from the log and look at it more closely:

- the **gap** between training and validation accuracy, which shows how much it overfits,
- the **confusion matrix** on the validation set, which shows which classes it mixes up,
- a comparison with the **linear classifier** from the previous notebooks, trained here with the same settings as there.

In [ ]:
from assignments import LinearClassifier
from utils import plot_confusion_matrix

best_name = max(experiments, key=lambda name: experiments[name]['val_acc'])
best_run = experiments[best_name]
best_model = best_run['model']

# The linear classifier with the settings from the linear classifier notebook
torch.random.manual_seed(69)
linear_classifier = LinearClassifier(num_features, num_classes, learning_rate=1e-2,
                                     batch_size=400, reg=0.1, num_iters=4000)
linear_classifier.train(X_train, y_train, X_val, y_val, verbose=False)

rows = [
    ("Linear classifier", linear_classifier),
    (f"MLP: {best_name}", best_model),
]
print(f"{'':36}{'train':>8}{'val':>8}{'gap':>8}")
for name, model in rows:
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    print(f"{name[:35]:36}{train_acc:>8.1%}{val_acc:>8.1%}{train_acc - val_acc:>8.1%}")

plot_confusion_matrix(y_val, best_model.predict(X_val), classes)

Notice:

- The MLP beats the linear classifier by more than 10 percentage points, with the same input pixels. The hidden layers help even on images.
- It also overfits much more (a gap of about 20 points instead of 2): the linear classifier has too few parameters to memorize 45,000 images, the MLP does not.
- In the confusion matrix, ship, plane and horse are recognized best, while cat and bird are the hardest. The most common mistakes are cats and dogs taken for each other, and cars and trucks taken for each other.

## An MLP Learns Many Templates

In the linear classifier notebook, every class had **one template**: a column of $\mathbf{W}$ reshaped to a 32×32×3 image. The first layer of the MLP computes $\mathbf{X}\mathbf{W}_1$ in exactly the same way, so every one of its $H_1$ neurons also has a 32×32×3 template. The deeper layers then combine the matches with these templates.

Below are the 10 templates of the linear classifier, followed by the first-layer templates of MLPs: the best run in the log, and two runs that differ **only in the regularization**, the 100-epoch run from above (`reg = 0.001`) and the same run with `reg = 0.01`. For every MLP we show the 20 neurons that the second layer relies on the most.

In [ ]:
from utils import plot_weights_as_templates, plot_neuron_templates

plot_weights_as_templates(linear_classifier.params['W'], classes)

# The 100-epoch run again, with 10x stronger regularization
strong_reg_name = "lr = 0.1, 100 epochs, reg = 0.01"
run_experiment(strong_reg_name, hidden_dim_1=200, hidden_dim_2=100, learning_rate=0.1,
               reg=1e-2, batch_size=200, num_epochs=100)


def template_title(name: str) -> str:
    """The run name, with its regularization strength unless the name already says it."""
    if "reg" in name:
        return f"MLP: {name}"
    return f"MLP: {name} (reg = {experiments[name]['hyperparameters']['reg']:g})"


# If the best run is the 100-epoch run, it is shown only once
plot_neuron_templates({
    template_title(best_name): best_model,
    template_title("lr = 0.1, 100 epochs"): experiments["lr = 0.1, 100 epochs"]['model'],
    template_title(strong_reg_name): experiments[strong_reg_name]['model'],
})

Notice:

- The linear classifier has to squeeze every class into one blurry template. The MLP has hundreds of templates, and they do not belong to a class: a template can capture a color blob, an edge between sky and ground, or the rough shape of a car or an animal, and several of them together describe a class.
- Compare the two runs that differ only in `reg`: with `reg = 0.01` the templates are **smoother** and easier to recognize, because the penalty on large weights removes the pixel noise. That network also overfits much less (training 54.3%, validation 50.3%, against 73.7% and 52.7% with `reg = 0.001`).
- The templates still compare pixels at fixed positions, like the linear classifier. An object at a different place or size needs a different template, which is why convolutional networks, which slide small templates over the image, do much better on images.

## Final Evaluation on the Test Set

Once you have finished tuning, choose the run with the best validation accuracy and evaluate it on the test set, **once**. If the test accuracy is much lower than the validation accuracy, you tuned too much to the 5,000 validation images.

In [ ]:
best_name = max(experiments, key=lambda name: experiments[name]['val_acc'])
best_run = experiments[best_name]

# Only now, and only once, look at the test set
test_acc = accuracy_score(y_test, best_run['model'].predict(X_test))

print(f"Chosen run:          {best_name}")
print(f"Hyperparameters:     {best_run['hyperparameters']}")
print(f"Validation accuracy: {best_run['val_acc']:.1%}")
print(f"Test accuracy:       {test_acc:.1%}")

## Your Report

Write a short report (200–400 words) in the cell below. The table from `show_experiments()` and the plots above are your evidence; refer to your runs by their names.

1. **Process.** Which runs did you do, and why did you choose each next one? Which strategy did you follow?
2. **Observations.** Which hyperparameters mattered most? Describe one failed run (what went wrong, and how you saw it in the curves) and your best run.
3. **Best configuration.** Its hyperparameters, training, validation and test accuracy, and the training time.
4. **Overfitting or underfitting?** Did your best model overfit or underfit, and how could you tell?
5. **Comparison.** How does your MLP compare to the linear classifier and k-NN, and why?

*Write your report here.*

## Conclusions and Next Steps

### How Far We Got on CIFAR-10

| Classifier | Test accuracy |
|---|---|
| Random guessing | 10% |
| k-NN | ~34% |
| Linear classifier | ~40% |
| MLP (this notebook) | ~50% |
| Convolutional networks | above 90% |
| State of the art | about 99% |

### Key Takeaways

1. **Hyperparameters decide whether a network learns at all.** The same MLP reaches about 50% or gets stuck, depending on the learning rate alone.
2. **Tune on the validation set, report on the test set.** The test set is used once, at the end.
3. **The training curves tell you what to change.** Slow decrease: a larger learning rate. Jumping loss: a smaller one. Training far above validation: more regularization.
4. **An MLP learns many templates**, and the deeper layers combine them. This is why it beats the linear classifier.

### Why Not Higher?

The MLP still treats an image as a list of 3072 numbers. It does not know which pixels are neighbors, and it needs a separate template for every position and size of an object. The next steps fix exactly this:

1. **Convolutional neural networks (CNNs)** slide small templates over the whole image, so an object is found wherever it appears.
2. **Data augmentation** shows the network shifted, flipped and zoomed copies of the training images.
3. **Better optimizers** such as Adam, and **batch normalization**, make deep networks easier to train.